# DocuMind — Phase 5A: LayoutLMv3 SROIE Fine-Tuning, Tracking & Deployment Readiness (GPU)

This notebook executes genuine reproducible SROIE preparation, bounded smoke verification, `microsoft/layoutlmv3-base` GPU fine-tuning with structured experiment tracking, held-out evaluation, Tesseract baseline comparison, checkpoint readiness verification (`import_checkpoint`), and archive export on Google Colab.

**Runtime:** Select *Runtime -> Change runtime type -> T4 GPU* before running.

In [ ]:
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq tesseract-ocr tesseract-ocr-eng zip libgl1 libglib2.0-0
!pip install -q torch torchvision transformers datasets seqeval accelerate pymupdf pytesseract opencv-python-headless pillow pydantic pydantic-settings python-dateutil scikit-learn

import shutil, torch, transformers, datasets
assert torch.cuda.is_available(), 'CUDA GPU runtime required! Enable T4 GPU via Runtime -> Change runtime type.'
free_gb = shutil.disk_usage('.').free / (1024 ** 3)
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB | Free disk: {free_gb:.1f} GB')
print(f'torch={torch.__version__}, transformers={transformers.__version__}, datasets={datasets.__version__}')
assert free_gb >= 8.0, f'Need at least 8 GB free disk space, found {free_gb:.1f} GB'

In [ ]:
import os
if not os.path.exists('DocuMind') and os.path.basename(os.getcwd()) != 'DocuMind':
    !git clone https://github.com/Abhiram-C-Divakaran/DocuMind.git
if os.path.basename(os.getcwd()) != 'DocuMind':
    %cd DocuMind
!git pull --ff-only
!git status -sb

## 1. Prepare Real SROIE Dataset Splits

Downloads `arvindrajan92/sroie_document_understanding` from Hugging Face Hub, persists real RGB receipt images, checks SHA-256 image hashes across splits to prevent leakage, aligns BIO entities (`COMPANY`, `DATE`, `ADDRESS`, `TOTAL`), and writes `data/sroie_processed/`.

In [ ]:
!python -m ml.data.prepare_dataset \
  --hf-dataset arvindrajan92/sroie_document_understanding \
  --output-dir data/sroie_processed \
  --seed 42

with open('data/sroie_processed/validation_report.md') as f:
    print(f.read())

## 1B. Bounded Pipeline Smoke Check (Isolated `/tmp/`)

Runs a fast 4-sample bounded training pass into `/tmp/` before full training to catch tokenization, CUDA AMP, or checkpoint-save issues early without polluting `checkpoints/` or `ml/experiments/`.

In [ ]:
import shutil
!python -m ml.training.train_layoutlmv3 \
  --data-dir data/sroie_processed \
  --output-dir /tmp/documind_smoke_ckpt \
  --best-model-dir /tmp/documind_smoke_best \
  --model-name microsoft/layoutlmv3-base \
  --epochs 1 \
  --batch-size 2 \
  --grad-accum 1 \
  --experiment-dir /tmp/documind_smoke_exp \
  --seed 42 \
  --smoke-test
shutil.rmtree('/tmp/documind_smoke_ckpt', ignore_errors=True)
shutil.rmtree('/tmp/documind_smoke_best', ignore_errors=True)
shutil.rmtree('/tmp/documind_smoke_exp', ignore_errors=True)
print('Bounded smoke test passed cleanly!')

## 2. Fine-Tune `microsoft/layoutlmv3-base`

Defaults to 5 epochs (`EPOCHS = 5`, configurable up to 8) with AdamW (`lr=2e-5`), linear warmup (`0.1`), gradient accumulation, CUDA AMP (`fp16`), early stopping (`patience=3`), and structured experiment tracking (`ml/experiments/latest_experiment.json`).

In [ ]:
EPOCHS = 5  # 5-8 epochs recommended for full convergence on T4/A100

!python -m ml.training.train_layoutlmv3 \
  --data-dir data/sroie_processed \
  --output-dir checkpoints/layoutlmv3-sroie \
  --best-model-dir checkpoints/layoutlmv3-sroie-best \
  --model-name microsoft/layoutlmv3-base \
  --epochs {EPOCHS} \
  --batch-size 4 \
  --grad-accum 2 \
  --lr 2e-5 \
  --patience 3 \
  --experiment-dir ml/experiments \
  --seed 42

import json
with open('ml/experiments/latest_experiment.json') as f:
    exp = json.load(f)
print('Experiment ID:', exp['experiment_id'])
print('Duration (s):', exp['duration_seconds'])
print('Best checkpoint:', exp['best_checkpoint'])

## 3. Held-Out Test Evaluation, Baseline Benchmark & Serving Verification

In [ ]:
!python -m ml.evaluation.evaluate \
  --checkpoint-dir checkpoints/layoutlmv3-sroie-best \
  --data-dir data/sroie_processed \
  --split test \
  --batch-size 4 \
  --output-dir ml/evaluation/results

!python -m ml.evaluation.compare_models \
  --checkpoint-dir checkpoints/layoutlmv3-sroie-best \
  --data-dir data/sroie_processed \
  --split test \
  --output-dir ml/evaluation/results \
  --run-full-ocr

!python -m ml.training.import_checkpoint \
  --checkpoint checkpoints/layoutlmv3-sroie-best

with open('ml/evaluation/results/evaluation_report.md') as f:
    print(f.read())

with open('ml/evaluation/results/benchmark_comparison.md') as f:
    print(f.read())

## 4. Export Best Checkpoint, Experiment Log & Reports

Archives inference weights (`checkpoints/layoutlmv3-sroie-best/`, excluding large optimizer `training_state.pt`), `ml/experiments/latest_experiment.json`, and `ml/evaluation/results/` for local import (`python -m ml.training.import_checkpoint --checkpoint documind_layoutlmv3_best.zip --target-dir checkpoints/layoutlmv3-sroie-best --apply-env`).

In [ ]:
!zip -r -q documind_layoutlmv3_best.zip checkpoints/layoutlmv3-sroie-best ml/experiments/latest_experiment.json ml/evaluation/results/evaluation_report.* ml/evaluation/results/benchmark_comparison.* -x "*/training_state.pt"
import os
print(f"Exported bundle size: {os.path.getsize('documind_layoutlmv3_best.zip') / (1024 ** 2):.2f} MB")
try:
    from google.colab import files
    files.download('documind_layoutlmv3_best.zip')
except ImportError:
    print('Saved archive at documind_layoutlmv3_best.zip')